# 04 학습 ② CLIP 검색기 (Colab, **GPU 필요: 런타임 → T4**)
처음 보는 음식 사진 → **비슷한 레시피를 직접 찾아서** 그 레시피들로 알레르기 비율을 계산한다 (판정은 그대로 룰).

| 파트 | 하는 일 | 보여주는 것 |
|---|---|---|
| 1~3 | 사진·레시피 짝 만들기, 음식 나누기 | 학습에 안 쓴 **"처음 보는 음식" 20종**을 따로 빼 둠 |
| 4 | 파인튜닝 **전** CLIP 성능 | 기준선 |
| 5 | 파인튜닝 (레시피 사진 ↔ 레시피 제목+재료) | 학습 ② 본체 |
| 6 | 파인튜닝 **후** 성능, 전/후 비교 | 학습 효과 |
| 7 | ConvNeXt vs CLIP (Food-101 정확도 · CPU 속도) | 왜 둘 다 쓰는지 |

**재는 것**: 음식 이름 맞히기(top1/top5)뿐 아니라 **알레르기 비율 오차**와 **"주의"를 "해당 없음"으로 놓친 비율** (우리 앱에서 제일 중요한 숫자).
처음부터 끝까지 30~50분. 결과는 `찍먹_실험기록/표/clip_결과.csv`, 모델은 `찍먹_실험기록/best/clip/`.

## 0. 드라이브 연결 · 코드 받기

In [ ]:
import os, sys, io, glob, math, time, random, unicodedata, subprocess
from google.colab import drive
drive.mount('/content/drive')

def find(parent, name):
    for d in os.listdir(parent):
        if unicodedata.normalize('NFC', d).strip() == name:
            return f'{parent}/{d}'
    raise FileNotFoundError(f"'{parent}' 안에 '{name}' 없음. 보이는 폴더: {sorted(os.listdir(parent))}")

TEAM = find('/content/drive/MyDrive', '26-2 딥러닝')
DATA = f'{TEAM}/data1'
IMG = f'{DATA}/images'                       # 03에서 모은 사진
RECIPES = f'{DATA}/table_recipes.parquet'    # 03에서 만든 레시피 (제목·재료)
EXP = find(TEAM, '찍먹_실험기록')
OUT, SAVE = f'{EXP}/표', f'{EXP}/best/clip'
os.makedirs(SAVE, exist_ok=True)

REPO = '/content/deeplearning_5'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yura103/deeplearning_5', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
sys.path.insert(0, REPO)

import torch
assert torch.cuda.is_available(), '런타임 → 런타임 유형 변경 → T4 GPU'
dev = 'cuda'
print('준비 완료', torch.cuda.get_device_name())

## 1. 사진 ↔ 레시피 짝 만들기
- 사진 한 장 = 레시피 한 개의 대표 사진. 짝이 되는 글 = **레시피 제목 + 재료 목록**.
- 03에서 같은 사진이 반복된 가짜(`_fake_md5.csv`)는 뺀다.

In [ ]:
import numpy as np, pandas as pd
from knowledge.rules import allergen_flags, ingredients_text
from knowledge.judge import level            # 비율 → 주의 / 확인 필요 / 해당 없음 (앱과 같은 기준)

recipes = pd.read_parquet(RECIPES, columns=['recipe_idx', 'label', 'title', 'NER']).drop_duplicates(['label', 'recipe_idx'])
recipes['ing'] = recipes.NER.map(ingredients_text)
recipes['text'] = (recipes.title.str.strip() + '. ' + recipes.ing).str.slice(0, 300)   # CLIP 글 길이 한계(77토큰)

fake = set(pd.read_csv(f'{IMG}/_fake_md5.csv').md5)
photos = pd.concat([pd.read_parquet(p, columns=['recipe_idx', 'label', 'jpg', 'md5']) for p in sorted(glob.glob(f'{IMG}/*.parquet'))])
photos = (photos[~photos.md5.isin(fake)].merge(recipes[['recipe_idx', 'label', 'text']], on=['recipe_idx', 'label'])
          .sample(frac=1, random_state=0).reset_index(drop=True))
print(f'사진 {len(photos):,}장, 음식 {photos.label.nunique()}종 | 레시피 {len(recipes):,}개, 음식 {recipes.label.nunique()}종')
print(photos.text.iloc[0])

## 2. 음식 나누기 — "처음 보는 음식"을 진짜로 처음 보게
- **처음 보는 음식 20종**: ConvNeXt가 모르는 새 후보(74종) 중 사진 30장 이상인 것에서 무작위. 이 음식 사진은 **학습에 한 장도 안 씀**.
  이 음식들의 레시피 글은 검색 대상에 있다 (실제 앱도 레시피는 미리 다 넣어 두니까).
- **아는 음식**: 나머지. 레시피 단위로 학습 80 / 검증 10 / 시험 10 (같은 레시피가 학습·시험에 동시에 안 들어가게).
- 사진 30장 미만(김밥·떡볶이 등)은 숫자가 흔들려서 학습·평가에서 빼고, 검색 대상(방해꾼)으로만 둔다.

In [ ]:
known = set(open(f'{REPO}/modeling/classes.txt').read().split())
cnt = photos.label.value_counts()
FEW = sorted(cnt[cnt < 30].index)
new_ok = sorted(c for c in cnt[cnt >= 30].index if c not in known)
UNSEEN = sorted(np.random.default_rng(0).choice(new_ok, size=20, replace=False))
SEEN = sorted(c for c in cnt[cnt >= 30].index if c not in UNSEEN)

h = (photos.recipe_idx.values * 2654435761) % 100          # 레시피 번호로 정해지는 고정 분할
photos['split'] = np.select([photos.label.isin(UNSEEN), ~photos.label.isin(SEEN), h < 80, h < 90],
                            ['unseen', 'none', 'train', 'val'], 'test')
train, val = photos[photos.split == 'train'], photos[photos.split == 'val']
ev = {'아는 음식': photos[photos.split == 'test'].groupby('label').head(30),
      '처음 보는 음식': photos[photos.split == 'unseen'].groupby('label').head(100)}

# 검색 대상: 175종 레시피 (음식당 최대 300개). 시험 사진의 원래 레시피는 빼서 "자기 글 찾기" 반칙 방지
held = set(pd.concat(ev.values()).recipe_idx)
index = (recipes[~recipes.recipe_idx.isin(held)].sample(frac=1, random_state=0)
         .groupby('label').head(300).reset_index(drop=True))
flags = allergen_flags(index.ing).reset_index(drop=True)       # 레시피 × 19항목 (룰)
ALLERGENS = list(flags.columns)
truth = pd.read_csv(f'{OUT}/food_table.csv', index_col='class')[ALLERGENS]   # 정답 = 그 음식 전체 레시피 통계

print('처음 보는 음식:', ', '.join(UNSEEN))
print(f'아는 음식 {len(SEEN)}종 | 학습 {len(train):,} · 검증 {len(val):,}장 | 사진 부족으로 제외 {len(FEW)}종: {", ".join(FEW)}')
print({k: len(v) for k, v in ev.items()}, f'| 검색 대상 레시피 {len(index):,}개')

## 3. CLIP 준비
`openai/clip-vit-base-patch32` (서버 CPU에서도 빠른 크기). 사진과 글을 같은 공간의 512차원 벡터로 바꾸고, 가까울수록 비슷하다.

In [ ]:
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms as T
from transformers import CLIPModel, CLIPTokenizer

NAME = 'openai/clip-vit-base-patch32'
tok = CLIPTokenizer.from_pretrained(NAME)
MEAN, STD = [0.4815, 0.4578, 0.4082], [0.2686, 0.2613, 0.2758]
BIC = T.InterpolationMode.BICUBIC
tf_eval = T.Compose([T.Resize(224, interpolation=BIC), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])
tf_train = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1), interpolation=BIC), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
load = lambda b: Image.open(io.BytesIO(b)).convert('RGB')

def img_feat(m, px):
    return F.normalize(m.visual_projection(m.vision_model(pixel_values=px).pooler_output), dim=-1)

def txt_feat(m, ids, mask):
    return F.normalize(m.text_projection(m.text_model(input_ids=ids, attention_mask=mask).pooler_output), dim=-1)

def tokens(texts):
    return tok(list(texts), padding=True, truncation=True, max_length=77, return_tensors='pt')

@torch.no_grad()
def emb_text(m, texts, bs=512):
    out = []
    for i in range(0, len(texts), bs):
        t = tokens(texts[i:i + bs]).to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(txt_feat(m, t.input_ids, t.attention_mask).float())
    return torch.cat(out)

@torch.no_grad()
def emb_img(m, jpgs, bs=256):
    out = []
    for i in range(0, len(jpgs), bs):
        px = torch.stack([tf_eval(load(b)) for b in jpgs[i:i + bs]]).to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(img_feat(m, px).float())
    return torch.cat(out)

## 4. 평가 함수 + 파인튜닝 전 성능
사진 한 장마다 두 가지로 맞혀 본다.
- **이름 맞히기**: "a photo of pad thai, a type of food" 같은 음식 이름 문장 175개 중 가장 가까운 것 (CLIP 기본 사용법)
- **레시피 검색** (우리 방식): 가장 가까운 레시피 `K`개 → 많이 나온 음식이 답. 그리고 그 `K`개 레시피로 **알레르기 비율**을 계산해
  정답 음식의 표 비율과 비교 → `알레르기_오차`(평균 절대 오차), `단계일치`(주의/확인/해당없음이 같은 비율), `주의놓침`(정답이 주의인데 해당 없음으로 나온 비율, 낮을수록 좋음)

In [ ]:
K = 30
NAMES = sorted(truth.index)
lv = np.vectorize(level)

def evaluate(m, tag):
    m.eval()
    Tx = emb_text(m, index.text.tolist())
    P = emb_text(m, [f'a photo of {c.replace("_", " ")}, a type of food.' for c in NAMES])
    lab = index.label.values
    Fl = torch.tensor(flags.values, dtype=torch.float32, device=dev)
    rows = []
    for name, d in ev.items():
        I, y = emb_img(m, d.jpg.tolist()), d.label.values
        zs = [[NAMES[j] for j in r] for r in (I @ P.T).topk(5).indices.tolist()]
        pred, est = [], []
        for c in range(0, len(I), 512):
            sim, idx = (I[c:c + 512] @ Tx.T).topk(K)
            est.append(Fl[idx].mean(1).cpu().numpy())
            for s_, i_ in zip(sim.tolist(), idx.tolist()):
                score = {}
                for s, j in zip(s_, i_):
                    score[lab[j]] = score.get(lab[j], 0) + s
                pred.append(sorted(score, key=score.get, reverse=True)[:5])
        est, tru = np.concatenate(est), truth.loc[y].values
        le, lt = lv(est), lv(tru)
        rows.append({'모델': tag, '시험': name, '장수': len(d),
                     '이름맞히기_top1': np.mean([p[0] == t for p, t in zip(zs, y)]),
                     '이름맞히기_top5': np.mean([t in p for p, t in zip(zs, y)]),
                     '레시피검색_top1': np.mean([p[0] == t for p, t in zip(pred, y)]),
                     '레시피검색_top5': np.mean([t in p for p, t in zip(pred, y)]),
                     '알레르기_오차': np.abs(est - tru).mean(),
                     '단계일치': (le == lt).mean(),
                     '주의놓침': (le[lt == '주의'] == '해당 없음').mean()})
    return pd.DataFrame(rows).round(3)

base = CLIPModel.from_pretrained(NAME).to(dev)
res_before = evaluate(base, 'CLIP 그대로')
del base; torch.cuda.empty_cache()
res_before

## 5. 파인튜닝 (학습 ②)
- 한 배치 128쌍에서 **맞는 사진-글 짝은 가깝게, 나머지 127개는 멀게** (CLIP 원래 학습 방식 그대로, contrastive loss).
- 이미 잘 배운 모델이라 학습률을 아주 작게(1e-5), 5에폭. 검증 손실이 제일 낮을 때를 저장.
- 한계: 같은 음식의 다른 레시피도 "틀린 짝"으로 취급된다. 그래도 제목·재료가 다르니 크게 문제되지 않음.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class Pairs(Dataset):
    def __init__(self, d, tf): self.j, self.t, self.tf = d.jpg.tolist(), d.text.tolist(), tf
    def __len__(self): return len(self.j)
    def __getitem__(self, i): return self.tf(load(self.j[i])), self.t[i]

def collate(b):
    px, txt = zip(*b)
    t = tokens(txt)
    return torch.stack(px), t.input_ids, t.attention_mask

def clip_loss(m, px, ids, mask):
    i, t = img_feat(m, px), txt_feat(m, ids, mask)
    logits = m.logit_scale.exp() * i @ t.T
    y = torch.arange(len(i), device=dev)
    return (F.cross_entropy(logits, y) + F.cross_entropy(logits.T, y)) / 2

EPOCHS, BS, LR = 5, 128, 1e-5
torch.manual_seed(0)
model = CLIPModel.from_pretrained(NAME).to(dev)
dl = DataLoader(Pairs(train, tf_train), BS, shuffle=True, drop_last=True, num_workers=2, collate_fn=collate)
vl = DataLoader(Pairs(val, tf_eval), BS, drop_last=True, num_workers=2, collate_fn=collate)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.1)
scaler = torch.amp.GradScaler('cuda')

best, log, t0 = 1e9, [], time.time()
for ep in range(EPOCHS):
    model.train(); tl = 0
    for px, ids, mask in dl:
        with torch.autocast('cuda', dtype=torch.float16):
            loss = clip_loss(model, px.to(dev), ids.to(dev), mask.to(dev))
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        model.logit_scale.data.clamp_(0, math.log(100))
        tl += loss.item()
    model.eval(); vls = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for px, ids, mask in vl:
            vls += clip_loss(model, px.to(dev), ids.to(dev), mask.to(dev)).item()
    tl, vls = tl / len(dl), vls / len(vl)
    log.append({'epoch': ep + 1, 'train_loss': tl, 'val_loss': vls})
    if vls < best:
        best = vls
        torch.save({k: v.half() for k, v in model.state_dict().items()}, f'{SAVE}/clip_ft.pt')
    print(f'[{ep+1}/{EPOCHS}] train {tl:.3f} | val {vls:.3f}{" ← 저장" if vls == best else ""} | {(time.time()-t0)/60:.0f}분')
pd.DataFrame(log).to_csv(f'{SAVE}/train_log.csv', index=False)

## 6. 파인튜닝 후 성능 · 전/후 비교

In [ ]:
model.load_state_dict({k: v.float() for k, v in torch.load(f'{SAVE}/clip_ft.pt').items()})
res_after = evaluate(model, 'CLIP 파인튜닝')
res = pd.concat([res_before, res_after]).sort_values(['시험', '모델'], ascending=[True, False])
res.to_csv(f'{OUT}/clip_결과.csv', index=False, encoding='utf-8-sig')
res

**앱에 쓸 검색 목록 저장**: 175종 레시피(음식당 최대 300개)를 파인튜닝된 CLIP으로 미리 벡터로 바꿔 둔다.
서버는 사진 한 장만 벡터로 바꾸고 이 목록과 곱하면 끝 (레시피 원문은 재배포 금지라 드라이브에만).

In [ ]:
app_index = recipes.sample(frac=1, random_state=0).groupby('label').head(300).reset_index(drop=True)
emb = emb_text(model, app_index.text.tolist()).half().cpu().numpy()
np.save(f'{SAVE}/index_emb.npy', emb)
app_index[['recipe_idx', 'label']].to_parquet(f'{SAVE}/index_meta.parquet', index=False)
print(f'검색 목록 {emb.shape} 저장 → {SAVE}')

## 7. ConvNeXt vs CLIP (Food-101 시험 사진, 음식당 20장)
- 둘 다 아는 101종에서 정확도 비교 + **서버 CPU에서 사진 한 장 처리 시간**.
- ConvNeXt는 101종 안에서만 답할 수 있고, CLIP은 레시피 글만 있으면 처음 보는 음식도 찾는다 → 아는 음식은 ConvNeXt(빠르고 정확), 모르는 음식은 CLIP.
- Food-101 압축 풀기에 5~10분 걸린다. 건너뛰려면 `RUN_A = False`.

In [ ]:
RUN_A = True
if RUN_A:
    from modeling.model import load_model, TF as TF_CNX
    F101 = '/content/food-101'
    if not os.path.exists(f'{F101}/meta/test.txt'):
        subprocess.run(['tar', 'xzf', f'{DATA}/food-101.tar.gz', '-C', '/content'], check=True)
    test = open(f'{F101}/meta/test.txt').read().split()
    by = {}
    for p in test:
        by.setdefault(p.split('/')[0], []).append(p)
    rng = random.Random(0)
    sub = [p for c in sorted(by) for p in rng.sample(by[c], 20)]
    ys = [p.split('/')[0] for p in sub]
    imgs = [Image.open(f'{F101}/images/{p}.jpg').convert('RGB') for p in sub]

    cnx, cnx_classes = load_model(f'{EXP}/best/convnext_tiny/best.pt', f'{REPO}/modeling/classes.txt')
    cnx = cnx.to(dev)
    with torch.no_grad():
        pc = []
        for i in range(0, len(imgs), 128):
            pc += cnx(torch.stack([TF_CNX(im) for im in imgs[i:i + 128]]).to(dev)).argmax(1).tolist()
    acc = {'ConvNeXt (학습 ①)': np.mean([cnx_classes[j] == y for j, y in zip(pc, ys)])}

    def clip101(m):
        P = emb_text(m, [f'a photo of {c.replace("_", " ")}, a type of food.' for c in cnx_classes])
        with torch.no_grad():
            I = torch.cat([img_feat(m, torch.stack([tf_eval(im) for im in imgs[i:i + 256]]).to(dev)) for i in range(0, len(imgs), 256)])
        return np.mean([cnx_classes[j] == y for j, y in zip((I @ P.T).argmax(1).tolist(), ys)])
    base = CLIPModel.from_pretrained(NAME).to(dev)
    acc['CLIP 그대로'] = clip101(base)
    acc['CLIP 파인튜닝'] = clip101(model)

    # CPU 속도: 서버처럼 사진 1장씩, 30번 평균
    torch.set_num_threads(os.cpu_count())
    def ms(fn, x):
        with torch.no_grad():
            for _ in range(3): fn(x)
            t = time.perf_counter()
            for _ in range(30): fn(x)
        return (time.perf_counter() - t) / 30 * 1000
    m_cpu, c_cpu = model.float().cpu().eval(), cnx.cpu().eval()
    speed = {'ConvNeXt (학습 ①)': ms(c_cpu, TF_CNX(imgs[0])[None]),
             'CLIP 그대로': None,
             'CLIP 파인튜닝': ms(lambda x: img_feat(m_cpu, x), tf_eval(imgs[0])[None])}   # 검색(행렬곱)은 1ms 안쪽이라 제외
    resA = pd.DataFrame({'Food-101 정확도(20장×101종)': acc, 'CPU ms/장': speed}).round(3)
    resA.to_csv(f'{OUT}/convnext_vs_clip.csv', encoding='utf-8-sig')
    model.to(dev)
    display(resA)

## 끝 — 결과 읽는 법
- `처음 보는 음식` 줄에서 **파인튜닝 후 `레시피검색_top1`·`단계일치`가 오르고 `주의놓침`이 내려가면** 학습 ②가 효과 있다는 증거.
- `이름맞히기` vs `레시피검색`: 이름만 비교하는 CLIP 기본 사용법보다 레시피를 직접 찾는 게 나은지.
- 7번: ConvNeXt가 아는 음식에선 더 정확·빠르면 → "아는 음식은 ConvNeXt, 모르는 음식은 CLIP" 구조의 근거.

다 돌리면 Claude에게 "04 다 돌렸어"라고 말하기. `clip_결과.csv`, `convnext_vs_clip.csv`를 읽고
서버(`backend/`)에 CLIP 검색을 붙인다 (2단계 RAG).